# 🖥️ Statistics Practice — `counts_annotated.tsv`

Lecture 6's count matrix, analysed with Lecture 9's tools.

- **Upload** `counts_annotated.tsv` first: `Insert → Files`, or drag it into the file panel on the left. It lands next to this notebook.
- **Runtime → Change runtime type → R**, then `Runtime → Run All`.

12 samples = 2 cell lines (**PC9**, **PC9GR**) × 2 treatments (**Control**, **Apatinib**) × 3 replicates — GSE129221, Song et al. 2019.

In [ ]:
# --- 1. load the matrix ----------------------------------------------------
counts <- read.delim("counts_annotated.tsv", row.names = 1, check.names = FALSE)

dim(counts)
head(counts[, 1:4])

# --- ggplot2 is not in Colab's R by default: install once, then load -------
if (!requireNamespace("ggplot2", quietly = TRUE)) install.packages("ggplot2", quiet = TRUE)
library(ggplot2)

## 1. The design

Column names carry the design: `PC9_Control_1` = cell line **PC9**, treatment **Control**, replicate **1**.

- 12 columns = 2 cell lines × 2 treatments × 3 replicates
- `Symbol` is column 1, so the samples are `colnames(counts)[-1]` and the matrix is `counts[, -1]`

In [ ]:
samples <- colnames(counts)[-1]

sample_design <- data.frame(
  sample    = samples,
  cell_line = sub("_.*", "", samples),
  treatment = sub("^[^_]+_([^_]+)_.*", "\\1", samples),
  replicate = as.integer(sub(".*_", "", samples))
)

sample_design

## 2. Library sizes

`colSums()` = total UMIs per sample.

| | |
|:--|:--|
| mean | 21,565,092 |
| median | 21,998,472 |
| sd | 970,173 |
| min | 19,485,055 (`PC9GR_Apatinib_3`) |
| max | 22,481,285 (`PC9_Control_1`) |

max − min = 3.0 M = **14% of the mean** → sequencing depth differs between samples. Compare a gene **within** a group, never raw counts across samples.

In [ ]:
counts_mat <- as.matrix(counts[, -1])

lib_sizes <- colSums(counts_mat)
lib_sizes

mean(lib_sizes)
median(lib_sizes)
sd(lib_sizes)
range(lib_sizes)

## 3. Filter the genes

A gene is testable only if it is counted in enough samples: **count ≥ 10 in ≥ 3 samples**.

39,376 genes → **17,860 kept**, 21,516 dropped.

In [ ]:
keep_genes  <- rowSums(counts_mat >= 10) >= 3
gene_counts <- counts_mat[keep_genes, ]
symbol_col  <- counts$Symbol[keep_genes]

sum(keep_genes)
nrow(gene_counts)

## 4. Descriptive statistics

Work on `log2(count + 1)`: counts run from 0 to 710,554, and the log compresses that. `+1` avoids `log(0)`.

Every sample: **mean 7.77–7.91**, **median 8.24–8.42**, **sd 2.79–2.96**.

**mean < median in all 12** → right-skewed: a few very high counts pull the mean up. Expected shape of expression data, not a sample-quality problem.

In [ ]:
log_counts <- log2(gene_counts + 1)

describe <- data.frame(
  mean   = round(colMeans(log_counts), 2),
  median = round(apply(log_counts, 2, median), 2),
  sd     = round(apply(log_counts, 2, sd), 2)
)

describe

## 5. Shape check — Shapiro-Wilk

`shapiro.test` needs **3 ≤ n ≤ 5000**, so take 5,000 genes from each sample.

| sample | W | p |
|:--|:--|:--|
| PC9_Control_1 | 0.9741 | 3.36e-29 |
| PC9_Apatinib_1 | 0.9755 | 1.82e-28 |
| PC9GR_Apatinib_3 | 0.9760 | 3.01e-28 |

p ≪ 0.05 → **not normal** in any sample. So: the t-test on raw counts is off; the t-test on `log2(count+1)` is close enough to use; the rank test is the honest check.

In [ ]:
shapiro.test(log_counts[1:5000, "PC9_Control_1"])
shapiro.test(log_counts[1:5000, "PC9_Apatinib_1"])
shapiro.test(log_counts[1:5000, "PC9GR_Apatinib_3"])

## 6. Sample × sample correlation

Spearman rho for all 144 pairs: a 12 × 12 matrix.

- off-diagonal: **min 0.924**, **max 0.997**
- PC9 vs PC9: 0.924–1.00 · PC9GR vs PC9GR: 0.971–1.00
- PC9 vs PC9GR: 0.924–0.961 → the cell line is the big split, not batch

Correlating samples is the QC step: a sample that does not correlate with its own replicates is the one to drop.

In [ ]:
cor_matrix <- round(cor(log_counts, method = "spearman"), 3)
cor_matrix

off_diag <- row(cor_matrix) != col(cor_matrix)
min(cor_matrix[off_diag])
max(cor_matrix[off_diag])

In [ ]:
cor_df <- data.frame(
  sample1 = rep(samples, times = length(samples)),
  sample2 = rep(samples, each  = length(samples)),
  rho     = as.vector(cor(log_counts, method = "spearman"))
)

ggplot(cor_df, aes(x = sample1, y = sample2, fill = rho)) +
  geom_tile() +
  geom_text(aes(label = round(rho, 2)), size = 2.2, colour = "black") +
  scale_fill_gradient(low = "white", high = "#d6336c", limits = c(0.9, 1)) +
  labs(x = "", y = "", title = "Spearman rho between the 12 samples") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 90, size = 6), axis.text.y = element_text(size = 6))

## 7. Per-gene test — PC9 Control vs PC9 Apatinib

n = 3 vs n = 3, one t-test per gene, on `log2(count + 1)`.

- a gene with **sd = 0 in both groups** makes `t.test` fail ("data are essentially constant") → return `NA`
- 17,860 filtered genes → 17,850 tests, **10 NA**
- raw p < 0.05: **3,647**

In [ ]:
control_idx  <- which(sample_design$treatment == "Control"  & sample_design$cell_line == "PC9")
apatinib_idx <- which(sample_design$treatment == "Apatinib" & sample_design$cell_line == "PC9")

p_values <- apply(log_counts, 1, function(g) {
  if (sd(g[control_idx]) == 0 && sd(g[apatinib_idx]) == 0) return(NA)
  suppressWarnings(t.test(g[control_idx], g[apatinib_idx])$p.value)
})

sum(is.na(p_values))
sum(p_values < 0.05, na.rm = TRUE)

## 8. Correcting for 17,850 tests

Bonferroni threshold: `0.05 / 17850 = 2.8e-6`. BH: rank the p-values, multiply the i-th by `17850 / i`, then carry the minimum forward.

| method | genes with p < 0.05 |
|:--|:--|
| raw | 3,647 |
| Bonferroni | 1 |
| BH | 2 |

3,645 of the raw hits are false-positive candidates. With n = 3 vs 3, correction leaves **two genes**.

In [ ]:
p_bf <- p.adjust(p_values, method = "bonferroni")
p_bh <- p.adjust(p_values, method = "BH")

0.05 / nrow(gene_counts)
sum(p_bf < 0.05, na.rm = TRUE)
sum(p_bh < 0.05, na.rm = TRUE)

## 9. The genes that pass

`log2FC = mean(log2 Apatinib) − mean(log2 Control)` — negative means **down** with the drug.

| Symbol | mean Control | mean Apatinib | log2FC | raw p | adj p (BH) |
|:--|:--|:--|:--|:--|:--|
| NCF4 | 67 | 32 | −1.04 | 8.61e-07 | 0.0154 |
| ANXA10 | 233 | 37 | −2.63 | 5.11e-06 | 0.0456 |

NCF4 is a NADPH-oxidase subunit (neutrophil); ANXA10 is an annexin. Both drop with Apatinib in PC9 — two genes out of 17,850, on n = 3.

In [ ]:
log2_fc <- rowMeans(log_counts[, apatinib_idx]) - rowMeans(log_counts[, control_idx])

top_genes <- data.frame(
  Symbol     = symbol_col,
  log2FC     = round(log2_fc, 2),
  mean_ctrl  = round(rowMeans(gene_counts[, control_idx]), 0),
  mean_apo   = round(rowMeans(gene_counts[, apatinib_idx]), 0),
  raw_p      = p_values,
  adj_p      = p_bh
)[order(p_bh), ]

head(top_genes, 6)

In [ ]:
volcano <- data.frame(Symbol = symbol_col, log2FC = log2_fc, adj_p = p_bh)
volcano <- volcano[!is.na(volcano$adj_p), ]
volcano$neg_log10 <- -log10(volcano$adj_p)

sig <- volcano[volcano$adj_p < 0.05, ]

ggplot(volcano, aes(x = log2FC, y = neg_log10)) +
  geom_point(size = 1, colour = "#98a3c7") +
  geom_hline(yintercept = -log10(0.05), linetype = 2, colour = "#e7ad52") +
  geom_vline(xintercept = c(-1, 1), linetype = 2, colour = "#e7ad52") +
  geom_point(data = sig, size = 2, colour = "#d6336c") +
  geom_text(data = sig, aes(label = Symbol), size = 2.6, vjust = 1.6, colour = "black") +
  labs(x = "log2 fold change (Apatinib - Control)", y = "-log10 adjusted p",
       title = "PC9 Control vs Apatinib (n = 3 vs 3)") +
  theme_minimal()

## 10. The rank test at n = 3

Wilcoxon, 3 vs 3, every gene:

- **0 genes** pass raw p < 0.05
- smallest p = **0.0593** · median p = **0.376**

With 3 vs 3 there are only 20 ways to split the ranks, so the exact two-sided p cannot go below 0.1 (values below 0.1 here come from ties, where R falls back to a normal approximation). **n = 3 has no power for the rank test.** The t-test "finds" genes only because it assumes normality.

In [ ]:
wilcox_p <- apply(log_counts, 1, function(g)
  suppressWarnings(wilcox.test(g[control_idx], g[apatinib_idx])$p.value))

sum(wilcox_p < 0.05, na.rm = TRUE)
min(wilcox_p, na.rm = TRUE)
median(wilcox_p, na.rm = TRUE)

## 11. PC9GR — same pipeline, more signal

| | PC9 | PC9GR |
|:--|:--|:--|
| NA (constant groups) | 10 | 21 |
| raw p < 0.05 | 3,647 | 5,421 |
| BH p < 0.05 | 2 | 374 |

374 genes pass in PC9GR, 2 in PC9 — the resistant line responds to Apatinib far more broadly, in the same data.

In [ ]:
control_gr  <- which(sample_design$treatment == "Control"  & sample_design$cell_line == "PC9GR")
apatinib_gr <- which(sample_design$treatment == "Apatinib" & sample_design$cell_line == "PC9GR")

p_gr <- apply(log_counts, 1, function(g) {
  if (sd(g[control_gr]) == 0 && sd(g[apatinib_gr]) == 0) return(NA)
  suppressWarnings(t.test(g[control_gr], g[apatinib_gr])$p.value)
})

sum(is.na(p_gr))
sum(p_gr < 0.05, na.rm = TRUE)
sum(p.adjust(p_gr, method = "BH") < 0.05, na.rm = TRUE)

## 12. Correlation and regression

Two genes that move together across all 12 samples: **AGPAT5 vs NCF4** (raw counts).

| | |
|:--|:--|
| Pearson r | 0.963 — p 4.99e-07 |
| Spearman rho | 0.968 — p 2.34e-07 |
| slope | 0.1563 |
| intercept | −48.07 |
| R² | 0.928 |

Reading: **AGPAT5 +100 UMIs → NCF4 +15.6 UMIs**, and the line explains **92.8%** of the variance. `r` is the strength, the slope is the size, R² is the fit.

No cause in any of it: both genes can be driven by the same sample effect (library depth, cell line).

In [ ]:
gene_a_row <- gene_counts[which(symbol_col == "AGPAT5"), ]
gene_b_row <- gene_counts[which(symbol_col == "NCF4"), ]

cor.test(gene_a_row, gene_b_row, method = "pearson")
suppressWarnings(cor.test(gene_a_row, gene_b_row, method = "spearman"))

fit <- lm(gene_b_row ~ gene_a_row)
coef(fit)
summary(fit)$r.squared

In [ ]:
scatter <- data.frame(AGPAT5 = gene_a_row, NCF4 = gene_b_row)

ggplot(scatter, aes(x = AGPAT5, y = NCF4)) +
  geom_point(size = 2, colour = "#98a3c7") +
  geom_abline(intercept = coef(fit)[1], slope = coef(fit)[2],
             colour = "#d6336c", linewidth = 0.4) +
  labs(title = paste("AGPAT5 vs NCF4:  r =", round(cor(gene_a_row, gene_b_row), 3),
                     "  R2 =", round(summary(fit)$r.squared, 3))) +
  theme_minimal()

## 13. Write the report

```r
write.csv(top_genes, "pc9_apatinib_results.csv", row.names = FALSE)
```

What a report states:

| | |
|:--|:--|
| data | GSE129221 — 39,376 genes, 12 samples, 17,860 filtered |
| design | 2 cell lines × 2 treatments × 3 replicates |
| test | t-test on `log2(count + 1)`, 3 vs 3, 17,850 tests |
| correction | BH, FDR 0.05 |
| result | 2 genes pass: **NCF4** (log2FC −1.04), **ANXA10** (−2.63) |
| caveat | n = 3, data not normal, Wilcoxon gives 0, BH keeps 2 |

In [ ]:
write.csv(top_genes, "pc9_apatinib_results.csv", row.names = FALSE)
list.files(pattern = "csv")